# 02 · Scorer and encoder baselines (E1, E3)
1. Load the thesis per-comment prediction files (exp1–exp8, seeds 42–44, val + test).
2. Check every file is row-aligned with the gold data (ids, gold sentiment, gold sarcasm, post).
3. Convert to the §5.2 format → `data/baselines/<model>_seed<s>_<split>.csv`.
4. Score all of them with the **one shared scorer** (`src/sarcllm/eval`) and check it reproduces ICARC-MS.
5. Bootstrap sanity check (post_id primary, post_group sensitivity) and detector routing statistics for the hybrid.

Test-set scores of the encoder baselines are already published (ICARC-MS), so scoring them here is not a new look at test.

## 1 · Setup

In [ ]:
import sys, re, json, datetime
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))           # use the shared scorer from src/

from sarcllm.eval.metrics import score, align, compute
from sarcllm.eval.bootstrap import ci_single, ci_paired
from sarcllm.eval.significance import mcnemar, sentiment_correct

PROC_DIR  = ROOT / "data" / "processed"
THESIS    = ROOT / "data" / "baselines" / "thesis_raw"
BASE_DIR  = ROOT / "data" / "baselines"
N_BOOT    = 10_000
pd.set_option("display.width", 200, "display.max_columns", 60)
print("thesis files:", len(list(THESIS.glob("*_preds.csv"))), "prediction CSVs")

## 2 · Gold data

In [ ]:
def load_gold(split):
    df = pd.read_json(PROC_DIR / f"{split}.jsonl", lines=True, dtype=False)
    for c in ("id", "post_id", "post_group"):
        df[c] = df[c].astype(str)
    return df

gold = {s: load_gold(s) for s in ("val", "test")}
for s, g in gold.items():
    print(f"{s:5s} {len(g):,} rows · {g.post_id.nunique()} posts · {g.post_group.nunique()} post_groups · "
          f"{g.is_sarcastic.sum():,} sarcastic")

## 3 · Find and name the thesis prediction files
`MODEL_NAMES` maps thesis experiment numbers to the names used in this project.

In [ ]:
MODEL_NAMES = {
    "exp1": "xlmr_base_sent",  "exp2": "xlmr_base_mt",  "exp5": "xlmr_base_hier",
    "exp6": "xlmr_large_sent", "exp7": "xlmr_large_mt", "exp8": "xlmr_large_hier",
}
pat = re.compile(r"^(exp\d+)_.*_s(\d+)_(test|val)_preds\.csv$")
files = []
for p in sorted(THESIS.glob("*_preds.csv")):
    m = pat.match(p.name)
    if not m:
        print("skipped (name not recognised):", p.name); continue
    exp, seed, split = m.groups()
    if exp not in MODEL_NAMES:
        print("skipped (exp not in MODEL_NAMES):", p.name); continue
    files.append({"exp": exp, "model": MODEL_NAMES[exp], "seed": int(seed), "split": split, "path": p})
inv = pd.DataFrame(files)
display(inv.pivot_table(index="model", columns=["split", "seed"], values="exp", aggfunc="count", fill_value=0))

## 4 · Alignment checks (STOP if any fail)
Every file must contain exactly the gold ids, and its `y_true` / `sarc_true` / `post_id` must equal our gold data.
If they do not, the thesis predictions belong to a different row order or data version.

In [ ]:
SENT_ID2LAB = {0: "negative", 1: "neutral", 2: "positive"}
preds, problems = {}, []

for f in files:
    df = pd.read_csv(f["path"], dtype={"id": str, "post_id": str})
    g = gold[f["split"]].set_index("id")
    key = (f["model"], f["seed"], f["split"])
    if set(df.id) != set(g.index) or df.id.duplicated().any():
        problems.append(f"{key}: id set differs from gold ({len(set(df.id) ^ set(g.index))} ids differ)"); continue
    gg = g.loc[df.id]
    checks = {
        "y_true":    (df.y_true.map(SENT_ID2LAB).values == gg.sentiment.values).mean(),
        "sarc_true": (df.sarc_true.values == gg.is_sarcastic.values).mean(),
        "post_id":   (df.post_id.values == gg.post_id.values).mean(),
    }
    for k, v in checks.items():
        if v < 1:
            problems.append(f"{key}: {k} agrees on only {v:.2%} of rows")
    preds[key] = df

print("\n".join(problems) or f"All {len(preds)} files aligned with gold (ids, labels, posts).")
assert not problems

## 5 · Convert to §5.2 format and save

In [ ]:
def to_contract(df):
    has_sarc = (df.sarc_pred != -1).any()
    return pd.DataFrame({
        "id": df.id,
        "sentiment_pred": df.y_pred.map(SENT_ID2LAB),
        "p_neg": np.nan, "p_neu": np.nan, "p_pos": np.nan,          # not saved by the thesis
        "is_sarcastic_pred": df.sarc_pred if has_sarc else np.nan,
        "p_sarcastic": np.nan,                                        # not saved -> HY2 needs a re-run
    })

baselines = {}
for (model, seed, split), df in preds.items():
    out = to_contract(df)
    assert out.sentiment_pred.notna().all(), f"{model} s{seed} {split}: unknown y_pred codes"
    if out.is_sarcastic_pred.notna().any():
        assert set(out.is_sarcastic_pred.unique()) <= {0, 1}, f"{model} s{seed}: sarc_pred not 0/1"
    out.to_csv(BASE_DIR / f"{model}_seed{seed}_{split}.csv", index=False)
    baselines[(model, seed, split)] = out

has_det = sorted({m for (m, s, sp), d in baselines.items() if d.is_sarcastic_pred.notna().any()})
print(f"wrote {len(baselines)} files to data/baselines/   ·   models with sarcasm predictions: {has_det}")

## 6 · Score every model × seed with the shared scorer

In [ ]:
KEY_METRICS = ["sent_f1_overall", "sent_f1_sarcastic", "sent_f1_nonsarcastic", "penalty",
               "neg_overpred_neu", "neg_overpred_pos", "det_f1_macro", "det_f1_sarcastic", "routing_recall"]

rows = []
for (model, seed, split), pr in baselines.items():
    m = score(gold[split], pr, by_script=False)
    rows.append({"model": model, "seed": seed, "split": split, **{k: m.get(k, np.nan) for k in KEY_METRICS}})
scores = pd.DataFrame(rows).sort_values(["split", "model", "seed"])

summary = (scores.groupby(["split", "model"])[KEY_METRICS].agg(["mean", "std"]))
display(scores[scores.split == "test"].set_index(["model", "seed"])[KEY_METRICS].round(4))
display(summary.loc["test"].round(4))

## 7 · Reproduce ICARC-MS (Dev Plan §13 — STOP if this fails)
Three-seed test means; tolerance ±0.0005.

In [ ]:
EXPECTED = {
    ("xlmr_large_sent", "sent_f1_overall"):   0.6461,
    ("xlmr_large_mt",   "sent_f1_overall"):   0.6501,
    ("xlmr_large_sent", "sent_f1_sarcastic"): 0.4544,
    ("xlmr_large_mt",   "sent_f1_sarcastic"): 0.4502,
    ("xlmr_large_mt",   "det_f1_macro"):      0.7547,
}
TOL = 5e-4
test_means = scores[scores.split == "test"].groupby("model")[KEY_METRICS].mean()
rep = pd.DataFrame([{"model": m, "metric": k, "ICARC-MS": v, "this scorer": test_means.loc[m, k]}
                    for (m, k), v in EXPECTED.items()])
rep["diff"] = rep["this scorer"] - rep["ICARC-MS"]
rep["ok"] = rep["diff"].abs() <= TOL
display(rep.round(4))
if rep.ok.all():
    print("Scorer reproduces ICARC-MS: PASS")
else:
    print("MISMATCH — compare with the thesis results.json below before going further.")

In [ ]:
# Cross-check against what the thesis itself saved in *_results.json (prints numeric fields only)
def flat(d, pre=""):
    out = {}
    for k, v in d.items():
        if isinstance(v, dict):
            out.update(flat(v, f"{pre}{k}."))
        elif isinstance(v, (int, float)) and not isinstance(v, bool):
            out[f"{pre}{k}"] = v
    return out

js = {}
for p in sorted(THESIS.glob("exp[678]_*_results.json")):
    m = re.match(r"^(exp\d+)_.*_s(\d+)_results\.json$", p.name)
    js[(MODEL_NAMES[m.group(1)], int(m.group(2)))] = flat(json.loads(p.read_text(encoding="utf-8")))
thesis_json = pd.DataFrame(js).T
display(thesis_json.round(4))

## 8 · Bootstrap sanity check
Paired post-level bootstrap, **multi-task large vs sentiment-only large**, per seed.
Primary unit = `post_id` (99 posts, as ICARC-MS); sensitivity unit = `post_group` (97).

In [ ]:
te = gold["test"]
ids = te.id.tolist()
units = {"post_id": te.post_id.to_numpy(), "post_group": te.post_group.to_numpy()}
CI_METRICS = ["sent_f1_overall", "sent_f1_sarcastic", "sent_f1_nonsarcastic", "penalty"]

ci_rows = []
for seed in (42, 43, 44):
    a = align(te, baselines[("xlmr_large_mt", seed, "test")], ids)
    b = align(te, baselines[("xlmr_large_sent", seed, "test")], ids)
    b.p_sarc = None
    for unit, pid in units.items():
        r = ci_paired(a, b, pid, n_boot=N_BOOT, seed=42)
        for k in CI_METRICS:
            ci_rows.append({"seed": seed, "unit": unit, "metric": k, **r.loc[k, ["estimate", "ci_lo", "ci_hi", "p_boot"]].to_dict()})
ci = pd.DataFrame(ci_rows)
display(ci.pivot_table(index=["metric", "seed"], columns="unit", values=["ci_lo", "ci_hi"]).round(4))
print("Width change post_group vs post_id (mean):",
      round((ci.query("unit=='post_group'").eval("ci_hi-ci_lo").mean()
             / ci.query("unit=='post_id'").eval("ci_hi-ci_lo").mean() - 1) * 100, 1), "%")

In [ ]:
# McNemar per seed (row-level sentiment correctness), as in ICARC-MS
mc = []
for seed in (42, 43, 44):
    a = align(te, baselines[("xlmr_large_mt", seed, "test")], ids)
    b = align(te, baselines[("xlmr_large_sent", seed, "test")], ids)
    sar = a.y_sarc == 1
    for name, mask in (("all", None), ("sarcastic", sar)):
        mc.append({"seed": seed, "subset": name, **mcnemar(sentiment_correct(a, mask), sentiment_correct(b, mask))})
display(pd.DataFrame(mc).round(4))

## 9 · Detector routing statistics (input to the hybrid)
The multi-task large detector decides which comments reach the LLM. Its sarcastic-class recall
caps what HY1 can fix. Validation numbers are what the pilot uses; test numbers are reported with HY1.

In [ ]:
route = []
for split in ("val", "test"):
    g = gold[split]
    flagged = {}
    for seed in (42, 43, 44):
        pr = baselines[("xlmr_large_mt", seed, split)]
        m = score(g, pr, by_script=True)
        flagged[seed] = set(pr.id[pr.is_sarcastic_pred == 1])
        route.append({"split": split, "seed": seed, "routed": len(flagged[seed]),
                      "routed_share": m["routed_share"], "routing_recall": m["routing_recall"],
                      **{f"recall_{s}": m.get(f"{s}/routing_recall") for s in ("sinhala", "latin", "mixed")}})
    union = set.union(*flagged.values()); inter = set.intersection(*flagged.values())
    route.append({"split": split, "seed": "union", "routed": len(union)})
    route.append({"split": split, "seed": "all 3", "routed": len(inter)})
display(pd.DataFrame(route).round(4))
print("'union' ≈ unique LLM calls needed for HY1 across the 3 seeds (the cache serves the rest).")

## 10 · Research-log entry

In [ ]:
tm = test_means
print(f"""## {datetime.date.today()} · Encoder baselines imported and scorer verified (notebook 02)
- Thesis prediction files: {len(preds)} (models {sorted({k[0] for k in preds})}, seeds 42–44, val+test); all aligned with gold.
- Labels only: thesis saved no probabilities → p_* and p_sarcastic empty. HY1 runs from labels; HY2 needs a multi-task re-run.
- Reproduction vs ICARC-MS: {'PASS' if rep.ok.all() else 'FAIL — see notebook 02 §7'} (max |diff| {rep['diff'].abs().max():.4f}).
- Test means: large_sent overall {tm.loc['xlmr_large_sent','sent_f1_overall']:.4f} / sarc {tm.loc['xlmr_large_sent','sent_f1_sarcastic']:.4f};
  large_mt overall {tm.loc['xlmr_large_mt','sent_f1_overall']:.4f} / sarc {tm.loc['xlmr_large_mt','sent_f1_sarcastic']:.4f} / det {tm.loc['xlmr_large_mt','det_f1_macro']:.4f};
  large_hier overall {tm.loc['xlmr_large_hier','sent_f1_overall']:.4f} / sarc {tm.loc['xlmr_large_hier','sent_f1_sarcastic']:.4f}.
- [DECISION proposed] Add E3 = exp8 hierarchical large as an encoder-only detect-then-resolve comparison for HY1.
- Bootstrap: {N_BOOT} resamples, seed 42; post_group unit changes CI width by the amount printed in §8.
""")